In [ ]:
# Cluster task. Connects to Lakebase with a Databricks-minted OAuth token (no stored password).
import sys, os

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
nb_dir = os.path.dirname(ctx.notebookPath().get())
if ("/Workspace" + nb_dir) not in sys.path:
    sys.path.append("/Workspace" + nb_dir)
import state_store as ss
from state_store import StateStore


def widget(name, default=""):
    dbutils.widgets.text(name, default)
    return dbutils.widgets.get(name)


db_instance = widget("db_instance", "lakefed-ingest")
db_name = widget("db_name", "lakefed_ingest")
store = StateStore.connect_lakebase(db_instance, db_name)

In [ ]:
# Execute lakebase_schema.sql (control, checkpoints, reseed queue, telemetry) idempotently.
# Statements: -- comments stripped first (they may contain ';'), then split on ';' (state_store,
# unit-tested). Safe for this controlled DDL (no ';' or '--' inside string literals).
schema_path = "/Workspace" + nb_dir + "/lakebase_schema.sql"
with open(schema_path) as f:
    statements = ss.split_sql_statements(f.read())

# One-time rename for instances created before the schema was renamed: move the old
# `lakefed_ingest_mt` schema (all tables + data) to `lakefed_ingest` before applying the DDL.
with store.connection.cursor() as cur:
    cur.execute("select nspname from pg_namespace where nspname in ('lakefed_ingest_mt', 'lakefed_ingest')")
    present = {r[0] for r in cur.fetchall()}
    if "lakefed_ingest_mt" in present and "lakefed_ingest" not in present:
        cur.execute("alter schema lakefed_ingest_mt rename to lakefed_ingest")
        print("renamed schema lakefed_ingest_mt -> lakefed_ingest")

# Guarded object renames (ct_checkpoint -> checkpoint, sweep_run -> ingest_run, CT columns -> generic
# positions, ...): only the ones still pending for this instance, so re-applying is a no-op. Data and
# constraints move with the renamed objects. Runs BEFORE the DDL so its create-if-not-exists
# statements find the renamed tables in place.
with store.connection.cursor() as cur:
    cur.execute("select c.relname from pg_class c join pg_namespace n on n.oid = c.relnamespace "
                "where n.nspname = %s", (ss.SCHEMA,))
    relations = {r[0] for r in cur.fetchall()}
    cur.execute("select table_name, column_name from information_schema.columns where table_schema = %s",
                (ss.SCHEMA,))
    columns = {(r[0], r[1]) for r in cur.fetchall()}
    renames = ss.rename_statements(relations, columns)
    for stmt in renames:
        cur.execute(stmt)
        print(stmt)

with store.connection.cursor() as cur:
    for stmt in statements:
        cur.execute(stmt)
print(f"applied {len(renames)} renames and {len(statements)} DDL statements to {db_name}")